PDF Chatbot using LLamaIndex

In [3]:
!pip install pypdf --quiet
!pip install chromadb llama-index llama-index-vector-stores-chroma llama-index-llms-ollama llama-index-embeddings-ollama --quiet
!pip install docling llama-index-readers-docling --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 86.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 104.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 89.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [4]:
import requests
import threading
import os, subprocess
import time
from pathlib import Path
from IPython.display import display, Markdown
import pypdf
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.readers.docling import DoclingReader
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

In [8]:
!sudo apt update --quiet
!sudo apt install -y pciutils --quiet
!sudo apt-get install zstd --quiet
!curl -fsSL https://ollama.com/install.sh | sh 

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
Building dependency tree...
Reading state information...
154 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'http

In [60]:
os.environ["OLLAMA_LOG_LEVEL"] = "error"

print("Starting Ollama in the system background...")

global_ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    close_fds=True
)

# Wait a brief moment for it to initialize
time.sleep(5)
print("🟢 Ollama is running quietly in the background. You can now move to the next cell!")

Starting Ollama in the system background...
🟢 Ollama is running quietly in the background. You can now move to the next cell!


In [12]:
%%capture
!ollama pull llama3.2
!ollama pull nomic-embed-text

In [13]:
!ollama list

]11;?\NAME                       ID              SIZE      MODIFIED       
nomic-embed-text:latest    0a109f422b47    274 MB    29 seconds ago    
llama3.2:latest            a80c4f17acd5    2.0 GB    35 seconds ago    


In [14]:
# Configure the LLM to use Ollama with the llama3.2 model and increase the timeout
Settings.llm = Ollama(model="llama3.2", request_timeout=360.0)

# Configure the embedding model to use Ollama with the nomic-embed-text model
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

print("LlamaIndex settings updated to use Ollama for LLM (llama3.2 with increased timeout) and embedding (nomic-embed-text).")

LlamaIndex settings updated to use Ollama for LLM (llama3.2 with increased timeout) and embedding (nomic-embed-text).


In [15]:
# Get the pdf
url = "https://www.nrb.org.np/contents/uploads/2026/03/Macroeconomic-Report-February-2026.pdf"
Path("data").mkdir(exist_ok=True)
local_filename = "./data/Macroeconomic-Report-February-2026.pdf"

# Send a GET request to the URL
response = requests.get(url)

# Check if the request was successful (Status Code 200)
if response.status_code == 200:
    # Open a local file in 'wb' (write binary) mode and save the content
    with open(local_filename, "wb") as file:
        file.write(response.content)
    print("Download complete!")
else:
    print(f"Failed to download. Status code: {response.status_code}")

Download complete!


In [16]:
# Load data
file_extractor = {".pdf": DoclingReader()}

documents = SimpleDirectoryReader(
    input_dir = "./data",
    file_extractor = file_extractor
).load_data()

[INFO] 2026-06-07 09:39:12,626 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-07 09:39:12,633 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-07 09:39:13,764 [RapidOCR] download_file.py:82: Download size: 4.53MB
[INFO] 2026-06-07 09:39:13,926 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-07 09:39:13,928 [RapidOCR] main.py:57: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-07 09:39:14,064 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-07 09:39:14,066 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4/cls/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-07 09:39:14,876 [

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

RapidOCR returned empty result!


In [17]:
for i, doc in enumerate(documents):
    print(f"--- Document {i+1} Text Snippet ---")
    print(doc.text[:500]) # View the first 1500 characters

--- Document 1 Text Snippet ---
## MACROECONOMIC REPORT

ANALYSIS AND OUTLOOK

FEBRUARY 2026

## NEPAL RASTRA BANK ECONOMIC RESEARCH DEPARTMENT

Central Office, Baluwatar, Kathmandu

## MACROECONOMIC REPORT ANALYSIS AND OUTLOOK FEBRUARY 2026

## Macroeconomic Report February 2026

## Nepal Rastra Bank

Economic Research Department

## Foreword

Nepal Rastra Bank (NRB), as the monetary authority of Nepal, has been formulating the monetary policy of Nepal as provisioned by the NRB Act, 2002. The formulation process has  evolved,


In [18]:
# 1. Initialize Persistent Storage
db = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = db.get_or_create_collection("rag_data_collection")

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Define chunking strategy
text_splitter = SentenceSplitter(chunk_size=256, chunk_overlap=20)

# 2. Check current database population size
current_count = chroma_collection.count()

if os.path.exists("./data") and len(os.listdir("./data")) > 0:
    print("Reading data directory...")
    # documents = SimpleDirectoryReader("./data").load_data()
    
    # CASE A: Database has data -> Only insert completely fresh documents
    if current_count > 0:
        print(f"🗄️ Existing database found ({current_count} records). Checking for new files...")
        
        # Gather all unique document source IDs currently living inside Chroma
        existing_metadatas = chroma_collection.get(include=['metadatas'])['metadatas']
        existing_doc_ids = {m.get('doc_id') for m in existing_metadatas if m}
        
        # Filter down the document array to raw sources not found in the DB
        new_documents = [doc for doc in documents if doc.doc_id not in existing_doc_ids]
        
        if new_documents:
            print(f"➕ Found {len(new_documents)} new/modified documents. Appending to collection...")
            
            # Use the clean index construction wrapper to handle the chunk-and-write loop safely
            for doc in new_documents:
                VectorStoreIndex.from_documents(
                    [doc],
                    storage_context=storage_context,
                    transformations=[text_splitter],
                    show_progress=False
                )
            print("🟢 New files successfully appended.")
        else:
            print("🙌 Everything is up to date! No new files to process.")
            
    # CASE B: Database is empty -> Build the baseline directly
    else:
        print("🆕 Database is completely empty. Building baseline index...")
        index = VectorStoreIndex.from_documents(
            documents,
            storage_context=storage_context,
            transformations=[text_splitter],
            show_progress=True
        )
        print("🟢 Baseline index successfully built and committed.")
else:
    print("⚠️ Warning: No files found in your data folder to extract.")

# 3. Instantiate the engine interface for querying
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, 
    storage_context=storage_context
)

print(f"🚀 System Ready. Total embedded records in DB: {chroma_collection.count()}")

Reading data directory...
🆕 Database is completely empty. Building baseline index...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/192 [00:00<?, ?it/s]

🟢 Baseline index successfully built and committed.
🚀 System Ready. Total embedded records in DB: 192


In [ ]:
# text_splitter = SentenceSplitter(chunk_size=256, chunk_overlap=20)

# try:
#     # Attempt to create the index
#     index = VectorStoreIndex.from_documents(
#         documents,
#         storage_context=storage_context,
#         transformations=[text_splitter],
#         show_progress=True
#     )
    
#     # This runs ONLY if the index was built successfully
#     print("🟢 Successfully indexed.")
    
# except Exception as e:
#     # This runs if an error occurred during indexing
#     print(f"🔴 Indexing failed.")
#     print(f"Error details: {e}")


In [20]:
!pip install llama-index-postprocessor-flashrank-rerank --quiet

In [65]:
from llama_index.core import set_global_handler
from llama_index.postprocessor.flashrank_rerank import FlashRankRerank
set_global_handler("simple")

# create a query engine and query
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=20,
    vector_store_query_mode="hybrid",  # Switches Chroma interface to Hybrid Mode
    alpha=0.4  # 0.5 balances semantic and keyword matches evenly
)

# configure response synthesizer
response_synthesizer = get_response_synthesizer()

similarity_filter = SimilarityPostprocessor()

reranker = FlashRankRerank(top_n=4)

# assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
    node_postprocessors=[similarity_filter, reranker],  # Runs filter first, then reranks the best ones
)

query = "How is remittance increasing from 2010?"
response = query_engine.query(query)
print(f"Q: {query}\n")
print(f"Response: ")
display(Markdown(response.response))

INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Q: How is remittance increasing from 2010?

Response: 


Remittances are steadily increasing over time, with a significant rise in their share of GDP. They have grown from negligible levels to 19.4 percent of GDP by 2010 and further to 28.2 percent by 2025, indicating a consistent increase in their volume. This growth is driven primarily by transfers, making the external balance reliant on sustained remittance inflows rather than exports of goods and services.

In [66]:
# Display the source nodes for references
print("\n--- Source Nodes (References) ---")
for i, node in enumerate(response.source_nodes):
    print(f"Source Node {i+1}:\n")
    print(node.get_content().strip()[:100]+"...")
    print("\n---------------------------------")


--- Source Nodes (References) ---
Source Node 1:

Remittances increased from  negligible levels to 19.4 percent of GDP in 2010 and further to 28.2

pe...

---------------------------------
Source Node 2:

With respect  to  the  FDI,  Nepal  has  opened  up its economy since the mid-1980s, but with played...

---------------------------------
Source Node 3:

Subsequent deficit in 1994/95 was  due  to  the  import  surges  following economic liberalization a...

---------------------------------
Source Node 4:

Travel-related services recorded a deficit: although tourism  income  increased  with  the  rise in ...

---------------------------------


In [26]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GOOGLE_API_KEY = user_secrets.get_secret("GOOGLE_API_KEY")

In [27]:
!pip install llama-index-llms-google-genai --quiet

In [28]:
# Import from the new google_genai module path
from llama_index.llms.google_genai import GoogleGenAI

# Set up your credentials
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

# Initialize the new class instead of Gemini()
llm = GoogleGenAI(model="models/gemma-4-31b-it")

INFO:httpx:HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"


ERROR: Could not find a version that satisfies the requirement llama_index.core.llama_dataset (from versions: none)
ERROR: No matching distribution found for llama_index.core.llama_dataset


In [36]:
import llama_index.core.evaluation as eval_mod

# Print everything inside the evaluation module
for item in dir(eval_mod):
    if not item.startswith("__"):  # Skip built-in python system properties
        print(item)


AnswerRelevancyEvaluator
BaseEvaluator
BaseRetrievalEvaluator
BatchEvalRunner
ContextRelevancyEvaluator
CorrectnessEvaluator
DatasetGenerator
EvaluationResult
FaithfulnessEvaluator
GuidelineEvaluator
HitRate
MRR
MultiModalRetrieverEvaluator
PairwiseComparisonEvaluator
QueryResponseDataset
QueryResponseEvaluator
RelevancyEvaluator
ResponseEvaluator
RetrievalEvalResult
RetrievalMetricResult
RetrieverEvaluator
SemanticSimilarityEvaluator
answer_relevancy
base
batch_runner
context_relevancy
correctness
dataset_generation
eval_utils
faithfulness
get_retrieval_results_df
guideline
notebook_utils
pairwise
relevancy
resolve_metrics
retrieval
semantic_similarity


In [54]:
import random
import time
from llama_index.core.evaluation import RetrieverEvaluator
import nest_asyncio

nest_asyncio.apply()

# 1. Sample 5 random nodes from your database corpus
sampled_nodes = random.sample(nodes, k=5)
print(f"🎯 Selected {len(sampled_nodes)} random nodes.")

# We will store the queries in a simple list of dictionaries
custom_qa_dataset = []

print("🧠 Generating text questions sequentially...")
for idx, node in enumerate(sampled_nodes):
    # Construct a direct prompt to ask Gemma to build targeted questions
    prompt = f"""
    Context text is provided below.
    ---------------------
    {node.text}
    ---------------------
    Given the context above, generate exactly 2 distinct user questions that can be answered explicitly using ONLY the information provided. 
    Provide each question on a new line. Do not number them or add introductory text.
    """
    
    try:
        # Use your GoogleGenAI model to generate the text
        response = llm.complete(prompt)
        
        # Split the output into individual question strings
        questions = [q.strip() for q in response.text.split("\n") if q.strip()]
        
        # Save them into our custom dataset array
        for question in questions[:2]: # Safety slice to keep exactly 2
            custom_qa_dataset.append({
                "query": question,
                "expected_node_id": node.id_
            })
            
        print(f"   Processed node [{idx+1}/5] -> Saved questions successfully.")
        
    except Exception as e:
        print(f" ⚠️ Skipping node generation step due to error: {e}")
        
    # Standard rate limit buffer for Gemma 4 (15 RPM)
    if idx < len(sampled_nodes) - 1:
        time.sleep(4.5)

print(f"✅ Custom Dataset Built! Created {len(custom_qa_dataset)} total questions.\n")



INFO:google_genai.models:AFC is enabled with max remote calls: 10.


🎯 Selected 5 random nodes.
🧠 Generating text questions sequentially...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


** Prompt: **

    Context text is provided below.
    ---------------------
    However, the pace of credit growth depends upon the overall business climate, especially the settlement of the current political transition  through  a  timely  election,  and the  formation  of  a  stable  government  to enhance  the  investors'  confidence.    Since the interest rate has been historically low,  resulting  in  low  aggregate  demand, increased public spending, a further boost in  the  imports,  and  an  increase  in  private sector confidence after the formation of a stable government post-election. Moreover,  growing  NPL  has  a  threefold impact: an increased number of blacklisted entrepreneurs  who  should  have  been  in additional investment demand, rising nonbanking assets of the banking system, and deteriorating capital adequacy. These three scenarios are, on the one hand, affecting the banking  system's  profitability  and  further dampening the credit demand on the other.
    --

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


...

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


** Prompt: **

    Context text is provided below.
    ---------------------
    Travel-related services recorded a deficit: although tourism  income  increased  with  the  rise in  the  number  of  tourists,  substantial outflows for education-related travel outweighed  these gains. The primary income account surplus declined, reflecting higher repatriation of dividends by  foreign  direct  investment  companies and increased interest payments on external loans.

The secondary income account has been in surplus in the recent years. It recorded a  substantial  surplus,  supported  by  a  39.1 percent rise in workers' remittances which

Figure 3.17: Exchange Rate Volatility

Financial account has remained in surplus, but with the restricted flows. The financial account reflects how financial capital flows into  and  out  of  Nepal  through  various investment and financial instruments. Given Nepal's partial capital account convertibility, the financial account is primarily reflected by 

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


** Prompt: **

    Context text is provided below.
    ---------------------
    The recent data shows improved recovery with stabilized non-performing loans. Substantial progress has been achieved in the infrastructure, products, and usage of payment systems,  with  the  digitalization of economic transactions.

The external sector remains robust, but  not  sustainable. The  external  sector indicators remain robust and have shown improvements.  The  trade  deficit  remains historically low. The export growth remains vulnerable due to its dependence on edible oil  export.  The  BoP  is  in  record surplus, reflected in growing foreign exchange (FOREX) reserves; but this surplus  is  largely  driven  by  remittance inflows, which itself is vulnerable to external  factors.  The  exchange  rate  peg has stabilized trade and investment flows with  India,  but  exchange  rate  with  other currencies remain volatile and continue to face depreciation pressure.

The  fiscal  sector  performan

KeyboardInterrupt: 

In [67]:
import time
from llama_index.core.schema import QueryBundle # Import the missing wrapper object

total_hits = 0
total_mrr = 0
evaluated_count = 0

print("🏃 Evaluating queries against your actual HYBRID + RERANK architecture...")
start_time = time.time()

for i, item in enumerate(custom_qa_dataset):
    query_text = item["query"]
    expected_id = item["expected_node_id"]
    
    # Locate the original reference node block
    original_node = next((n for n in nodes if n.id_ == expected_id), None)
    if not original_node:
        continue
    expected_text = original_node.text[:100] # Grab first 100 chars for matching
    
    try:
        # CRUCIAL FIX: Wrap the plain query string inside a formal QueryBundle object!
        query_bundle = QueryBundle(query_str=query_text)
        
        # Pass the bundle to your query engine setup
        retrieval_response = query_engine.retrieve(query_bundle)
        
        # Extract the final node text results
        retrieved_texts = [node.node.get_content() for node in retrieval_response]
        
        # Text-Based Evaluation Logic
        hit = 0
        mrr_score = 0.0
        
        for rank, text in enumerate(retrieved_texts):
            if expected_text in text:
                hit = 1
                mrr_score = 1.0 / (rank + 1)
                break
        
        total_hits += hit
        total_mrr += mrr_score
        evaluated_count += 1
        
        print(f"   [{evaluated_count}/{len(custom_qa_dataset)}] Text-Hit: {hit} | MRR: {mrr_score:.2f}")
        
    except Exception as e:
        print(f" ⚠️ Error processing item {i}: {e}")

end_time = time.time()
print(f"\n⏱️ Evaluation finished cleanly in {end_time - start_time:.2f} seconds.")

# Print real-world production performance metrics
if evaluated_count > 0:
    print("\n===== TRUE PIPELINE PERFORMANCE =====")
    print(f"📊 Actual Hit Rate: {total_hits / evaluated_count:.2f}")
    print(f"🎯 Actual MRR:      {total_mrr / evaluated_count:.2f}")
    print("========================================")
else:
    print("❌ Evaluation failed to execute.")

INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


🏃 Evaluating queries against your actual HYBRID + RERANK architecture...
   [1/10] Text-Hit: 1 | MRR: 1.00


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


   [2/10] Text-Hit: 1 | MRR: 1.00


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


...

   [5/10] Text-Hit: 1 | MRR: 0.25
   [6/10] Text-Hit: 0 | MRR: 0.00


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


   [7/10] Text-Hit: 1 | MRR: 1.00
   [8/10] Text-Hit: 1 | MRR: 1.00


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


   [9/10] Text-Hit: 1 | MRR: 1.00
   [10/10] Text-Hit: 1 | MRR: 0.50

⏱️ Evaluation finished cleanly in 1.27 seconds.

===== TRUE PIPELINE PERFORMANCE =====
📊 Actual Hit Rate: 0.80
🎯 Actual MRR:      0.68


In [58]:
print(custom_qa_dataset)

[{'query': 'What factors does the pace of credit growth depend upon?', 'expected_node_id': 'ead30670-05c3-4e70-905a-769304442b39'}, {'query': 'What is the threefold impact of growing NPLs?', 'expected_node_id': 'ead30670-05c3-4e70-905a-769304442b39'}, {'query': 'What factors does the foreign exchange market pressure index (EMPI) use to assess pressure on the external sector?', 'expected_node_id': 'ec5631e1-0ff0-44ba-9218-19a778349446'}, {'query': 'Why has the growth in time deposits declined or turned negative since 2024?', 'expected_node_id': 'ec5631e1-0ff0-44ba-9218-19a778349446'}, {'query': 'On what page is the "Key Timeline of the Report and Alignments to Monetary Policy" located?', 'expected_node_id': 'e70f7001-a861-4362-9127-ed17d6802a42'}, {'query': 'What is the section number for the "Monetary Policy Framework of the Nepal Rastra Bank"?', 'expected_node_id': 'e70f7001-a861-4362-9127-ed17d6802a42'}, {'query': 'Why did travel-related services record a deficit despite an increase 

In [69]:
import random
import time

sampled_nodes = random.sample(nodes, k=5)
print(f"🎯 Selected {len(sampled_nodes)} random nodes.")

custom_qa_dataset = []

print("🧠 Generating Q&A Pairs with Reference Answers...")
for idx, node in enumerate(sampled_nodes):
    prompt = f"""
    Context text is provided below.
    ---------------------
    {node.text}
    ---------------------
    Given the context above, generate exactly 2 distinct user questions along with their concise reference answers.
    Format your response EXACTLY like this for each pair, with no other text:
    Q: [Your question here]
    A: [Your concise reference answer here]
    """
    
    try:
        response = llm.complete(prompt)
        lines = [line.strip() for line in response.text.split("\n") if line.strip()]
        
        current_q = None
        for line in lines:
            if line.startswith("Q:"):
                current_q = line[2:].strip()
            elif line.startswith("A:") and current_q:
                custom_qa_dataset.append({
                    "query": current_q,
                    "reference_answer": line[2:].strip(),
                    "expected_node_id": node.id_
                })
                current_q = None
                
        print(f"   Processed node [{idx+1}/5]")
    except Exception as e:
        print(f" ⚠️ Error generating pair: {e}")
        
    if idx < len(sampled_nodes) - 1:
        time.sleep(5) # Rate limit safety cushion

print(f"✅ Dataset built with {len(custom_qa_dataset)} fully grounded reference pairs!")

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


🎯 Selected 5 random nodes.
🧠 Generating Q&A Pairs with Reference Answers...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


** Prompt: **

    Context text is provided below.
    ---------------------
    (2025, January). The welfare cost of inflation in production networks. Paper  presented  at  the  2025  Annual  Meeting  of  the  American  Economic Association, San Francisco, CA.
- Aguiar,  M.,  &amp;  Bils,  M.  (2015).  Has  Consumption  Inequality  Mirrored  Income  Inequality? American Economic Review , 105(9), 2725-2756. https://doi.org/10.1257/aer.20120599
- Athukorala, P., &amp; Wagle, S. (2022). The sovereign debt crisis in Sri Lanka: causes, policy response and prospects. New York: United Nations Development Programme, 21.
- Borio,  C.  (2023)  Getting  up  from  the  floor  (BIS  Working  Papers  No.  1100).  Bank  for International Settlements.
- Budha, B. B. (2026).
    ---------------------
    Given the context above, generate exactly 2 distinct user questions along with their concise reference answers.
    Format your response EXACTLY like this for each pair, with no other text:
    Q: [Yo

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


...

** Prompt: **

    Context text is provided below.
    ---------------------
    During the  second  half  of  2025/26,  the  export grew  by  43.8  percent,  however,  when Soybean  oil  export  is  excluded,  export growth drops sharply to just 5.7 percent, indicating a stagnation in traditional and non-traditional export sectors.  Import dynamics reinforce this assessment through  a  pattern of  matching  trade, with crude soybean oil accounting for 6.1 percent  share  of  total  imports.  Imports grew by 14.2 percent in the review period. Besides, crude Soybean oil, import growth is led by chemical fertilizers, gold, transport equipment, vehicles and spare parts, and telecommunication equipment and  parts,  among  others. The  services account  surplus  narrowed,  mainly  due to weaker performance in the transport sector  as  higher  trade  volumes  raised transportation costs.
    ---------------------
    Given the context above, generate exactly 2 distinct user questions along w

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


** Prompt: **

    Context text is provided below.
    ---------------------
    On the supply side, BFIs have seen a rise in their NPLs over recent years, thereby affecting their non-banking assets, capital and profitability. The focus of the BFIs is thus on loan recovery, rather than expanding the business. On the demand side, one of the reasons could be attributed to the significant increase in the number of blacklisted individuals and companies. Similarly, a plunge in aggregate demand with slowed down economic activities, and a low business confidence is also equally responsible.   Furthermore, poor fiscal performance, especially capital spending, further contributed to this slow-down. The government spending experienced a negative growth in 2023/24, leading to fiscal contraction. The decrease in imports and production observed in those years also caused a decrease in the aggregate demand, whose aftereffects are still palpable. The Gen Z protests in 2025 could have further deterior

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


** Prompt: **

    Context text is provided below.
    ---------------------
    ...........................     | 49    |
|                                                                                                                                   | 4.7                                                                                                                               | Key Assumptions of the Projections/Outlook ............................................................                               | 49    |
|                                                                                                                                   | 4.8                                                                                                                               | Risks to the Outlook .................................................................................................
    ---------------------
    Given the context above, generate exactly 2 distinct

In [70]:
import time
from llama_index.core.schema import QueryBundle

# Tracking variables
total_faithfulness_hits = 0
total_relevancy_score = 0
total_correctness_score = 0
evaluated_count = 0

print("🏃 Evaluating RAG Triad (Faithfulness, Relevancy, Correctness)...")
start_time = time.time()

for i, item in enumerate(custom_qa_dataset):
    query_text = item["query"]
    reference_answer = item["reference_answer"]
    
    try:
        # 1. Generate the answer from your production RAG system
        response_obj = query_engine.query(QueryBundle(query_str=query_text))
        generated_answer = response_obj.response
        
        # Extract the source chunks your retriever actually found
        source_contexts = "\n---\n".join([node.node.get_content() for node in response_obj.source_nodes])
        
        print(f"\n📝 [{i+1}/{len(custom_qa_dataset)}] User Query: {query_text}")
        print(f"🤖 RAG Generated Answer: {generated_answer}")
        print(f"📖 Ground Truth Reference: {reference_answer}")
        
        # --- CHECK 1: FAITHFULNESS (Anti-Hallucination) ---
        faithfulness_prompt = f"""
        You are an expert judge. Review the context and the answer below.
        
        Context:
        {source_contexts}
        
        Answer:
        {generated_answer}
        
        Does the answer contain information NOT found in the context or unsupported by it? 
        Respond with exactly one word: "HALLEUCINATED" or "FAITHFUL". Do not explain.
        """
        time.sleep(5) # Pacing API rate limit
        faith_res = llm.complete(faithfulness_prompt)
        verdict = faith_res.text.strip().upper()
        
        is_faithful = 1 if "FAITHFUL" in verdict else 0
        total_faithfulness_hits += is_faithful
        
        # --- CHECK 2: ANSWER RELEVANCY (Focus) ---
        relevancy_prompt = f"""
        You are an expert judge. Is the Generated Answer relevant to the User Query? 
        It does not have to be factually correct, but it must directly address the question asked without rambling.
        
        User Query: {query_text}
        Generated Answer: {generated_answer}
        
        Output a single float score between 0.0 (completely irrelevant) and 1.0 (perfectly on topic) on the first line. 
        Do not write any introductory or explanatory text.
        """
        time.sleep(5) 
        rel_res = llm.complete(relevancy_prompt)
        relevancy_score = float(rel_res.text.strip().split("\n")[0])
        total_relevancy_score += relevancy_score
        
        # --- CHECK 3: CORRECTNESS (Accuracy vs Reference) ---
        correctness_prompt = f"""
        You are an expert judge. Compare the Generated Answer against the trusted Reference Answer.
        
        User Query: {query_text}
        Reference Answer: {reference_answer}
        Generated Answer: {generated_answer}
        
        Score the correctness based on these guidelines:
        1.0 - Completely incorrect or irrelevant.
        3.0 - Mostly correct but missing key factual metrics or details.
        5.0 - Fully correct and perfectly matches the core facts of the reference.
        
        Output a single float score between 1.0 and 5.0 on the first line. Do not write anything else.
        """
        time.sleep(5) 
        corr_res = llm.complete(correctness_prompt)
        correctness_score = float(corr_res.text.strip().split("\n")[0])
        total_correctness_score += correctness_score
        
        evaluated_count += 1
        
        # Individual metric logging
        print(f"⚖️ Faithfulness: {'🟢 PASS' if is_faithful else '🔴 HALLUCINATED'}")
        print(f"🎯 Relevancy:     {relevancy_score * 100:.1f}%")
        print(f"📊 Correctness:   {correctness_score:.1f} / 5.0")
        
    except Exception as e:
        print(f" ⚠️ Skipping item {i} due to grading error: {e}")
        
    print("-" * 60)
    time.sleep(5) # Cooldown buffer between items

end_time = time.time()
print(f"\n⏱️ Complete Evaluation finished in {end_time - start_time:.2f} seconds.")

# --- COMPUTE PERCENTAGES AND SCORECARD ---
if evaluated_count > 0:
    faithfulness_pct = (total_faithfulness_hits / evaluated_count) * 100
    relevancy_pct = (total_relevancy_score / evaluated_count) * 100
    
    # Map 1.0 -> 5.0 scale to a 0% -> 100% scale for linear grading visibility
    # Formula: ((score - min) / (max - min)) * 100
    avg_correctness_score = total_correctness_score / evaluated_count
    correctness_pct = ((avg_correctness_score - 1.0) / (5.0 - 1.0)) * 100
    
    print("\n==================================================")
    print("🏆       FINAL SYSTEM PERFORMANCE SCORECARD       🏆")
    print("==================================================")
    print(f"🟢 Faithfulness Rate (No Hallucination): {faithfulness_pct:.1f}%")
    print(f"🎯 Mean Answer Relevancy Score:         {relevancy_pct:.1f}%")
    print(f"📊 Mean Fact Correctness Score:         {avg_correctness_score:.2f} / 5.0 ({correctness_pct:.1f}%)")
    print("==================================================")
    
    # Quick Diagnostic Insight
    print("\n💡 Quick Assessment:")
    if faithfulness_pct < 80.0:
        print("   -> Your system is hallucinating. Try adding system prompt constraints like: 'Answer using ONLY the provided text.'")
    if correctness_pct < 75.0:
        print("   -> Your model is missing context details. Consider tweaking chunk sizes or expanding your similarity retrieval window.")
    if faithfulness_pct >= 80.0 and correctness_pct >= 75.0:
        print("   -> Looking incredibly sharp! Your RAG generation pipeline behaves highly reliably.")
else:
    print("❌ No items were successfully evaluated.")

🏃 Evaluating RAG Triad (Faithfulness, Relevancy, Correctness)...


INFO:httpx:HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



📝 [1/10] User Query: Who authored the 2022 report on the sovereign debt crisis in Sri Lanka?
🤖 RAG Generated Answer: P. Athukorala and S. Wagle co-authored the 2022 report on the sovereign debt crisis in Sri Lanka.
📖 Ground Truth Reference: Athukorala, P., and Wagle, S.


INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


** Prompt: **

        You are an expert judge. Review the context and the answer below.
        
        Context:
        (2025, January). The welfare cost of inflation in production networks. Paper  presented  at  the  2025  Annual  Meeting  of  the  American  Economic Association, San Francisco, CA.
- Aguiar,  M.,  &amp;  Bils,  M.  (2015).  Has  Consumption  Inequality  Mirrored  Income  Inequality? American Economic Review , 105(9), 2725-2756. https://doi.org/10.1257/aer.20120599
- Athukorala, P., &amp; Wagle, S. (2022). The sovereign debt crisis in Sri Lanka: causes, policy response and prospects. New York: United Nations Development Programme, 21.
- Borio,  C.  (2023)  Getting  up  from  the  floor  (BIS  Working  Papers  No.  1100).  Bank  for International Settlements.
- Budha, B. B. (2026).
---
The issue, however, is  the  need  for  some  credible  ways  in managing the NPL, like the establishment of  a  separate  institution  envisioned  in  the

3 Approximately, the popula

...

** Prompt: **

        You are an expert judge. Review the context and the answer below.
        
        Context:
        The  Board  of  Directors  chaired by  the  Governor  is  the  ultimate  decisionmaking  body  for  policy  formulation.  The policy's draft is prepared by the Economic Research Department while suggestions are being sought from other NRB departments and stakeholders. The suggestions and monetary policy stances are first discussed at the Interdepartmental Coordination Committee at the Economic Research Department.  The  draft  of  the  monetary policy, both annual and quarterly reviews, is then presented to the  Management Committee  of  the NRB  chaired  by the Governor.  The  committee  then  forwards the  draft  to  the  Board  for  final  decisions.

There  is  also  a  Monetary  Policy  Advisory Committee, chaired by the Deputy Governor,  comprising  two  external  expert members and head of the NRB's Economic Research Department, which advises on  the  moneta

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


** Prompt: **

        You are an expert judge. Is the Generated Answer relevant to the User Query? 
        It does not have to be factually correct, but it must directly address the question asked without rambling.
        
        User Query: What is the subject of section 4.8?
        Generated Answer: Unfortunately, I don't have any information about a section 4.8 in the provided context. The given text seems to be divided into sections with headings such as "Policy Framework", "Industrial activity", and others, but there is no mention of a section 4.8. Therefore, I cannot provide an answer to this query based on the available information.
        
        Output a single float score between 0.0 (completely irrelevant) and 1.0 (perfectly on topic) on the first line. 
        Do not write any introductory or explanatory text.
        
**************************************************
** Completion: **
1.0
**************************************************




INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


** Prompt: **

        You are an expert judge. Compare the Generated Answer against the trusted Reference Answer.
        
        User Query: What is the subject of section 4.8?
        Reference Answer: Risks to the Outlook.
        Generated Answer: Unfortunately, I don't have any information about a section 4.8 in the provided context. The given text seems to be divided into sections with headings such as "Policy Framework", "Industrial activity", and others, but there is no mention of a section 4.8. Therefore, I cannot provide an answer to this query based on the available information.
        
        Score the correctness based on these guidelines:
        1.0 - Completely incorrect or irrelevant.
        3.0 - Mostly correct but missing key factual metrics or details.
        5.0 - Fully correct and perfectly matches the core facts of the reference.
        
        Output a single float score between 1.0 and 5.0 on the first line. Do not write anything else.
        
********